In [1]:
import torch
import numpy as np
import os

DATA_DIR = r"C:\DTI_GNN\data"
EVAL_DIR = r"C:\DTI_GNN\evaluation"

# ------------------------------------------------------------
# Load training data
# ------------------------------------------------------------

train_data = torch.load(
    os.path.join(DATA_DIR, "train_data_v2.pt"),
    weights_only=False
)

train_y = np.array([
    float(d.y.item())
    for d in train_data
])

# ------------------------------------------------------------
# Load cold-start data
# ------------------------------------------------------------

cold_drug = torch.load(
    os.path.join(EVAL_DIR, "cold_drug_data_v2.pt"),
    weights_only=False
)

cold_pair = torch.load(
    os.path.join(EVAL_DIR, "cold_pair_data_v2.pt"),
    weights_only=False
)

cold_target = torch.load(
    os.path.join(EVAL_DIR, "cold_target_data_v2.pt"),
    weights_only=False
)


# ------------------------------------------------------------
# Function
# ------------------------------------------------------------

def inspect_dataset(name, data):

    y = np.array([
        float(d.y.item())
        for d in data
    ])

    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

    print("Number of samples :", len(y))
    print("Minimum y         :", y.min())
    print("Maximum y         :", y.max())
    print("Mean y            :", y.mean())
    print("Median y          :", np.median(y))
    print("Std y             :", y.std())

    print("\nFirst 20 y values:")
    print(y[:20])

    print("\nPercentiles:")
    print("1%   :", np.percentile(y, 1))
    print("5%   :", np.percentile(y, 5))
    print("25%  :", np.percentile(y, 25))
    print("50%  :", np.percentile(y, 50))
    print("75%  :", np.percentile(y, 75))
    print("95%  :", np.percentile(y, 95))
    print("99%  :", np.percentile(y, 99))


# ------------------------------------------------------------
# Training distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TRAINING DATA")
print("=" * 70)

print("Number of samples :", len(train_y))
print("Minimum y         :", train_y.min())
print("Maximum y         :", train_y.max())
print("Mean y            :", train_y.mean())
print("Median y          :", np.median(train_y))
print("Std y             :", train_y.std())


# ------------------------------------------------------------
# Cold-start distributions
# ------------------------------------------------------------

inspect_dataset(
    "COLD-DRUG",
    cold_drug
)

inspect_dataset(
    "COLD-PAIR",
    cold_pair
)

inspect_dataset(
    "COLD-TARGET",
    cold_target
)


TRAINING DATA
Number of samples : 569030
Minimum y         : 3.0
Maximum y         : 12.698969841003418
Mean y            : 6.7035064357907865
Median y          : 6.718966484069824
Std y             : 1.3538411364648653

COLD-DRUG
Number of samples : 138556
Minimum y         : 3.0
Maximum y         : 11.90308952331543
Mean y            : 6.615518415570277
Median y          : 6.623423099517822
Std y             : 1.4113094134778377

First 20 y values:
[7.15366268 7.58502674 6.27572393 5.         7.22184896 5.68193674
 5.         5.46218109 4.60205984 6.21467018 6.82390881 6.79587984
 6.92081881 7.22184896 5.92811823 7.04575729 7.         5.
 5.         7.04575729]

Percentiles:
1%   : 3.586624777317047
5%   : 4.318758964538574
25%  : 5.525783538818359
50%  : 6.623423099517822
75%  : 7.6575775146484375
95%  : 8.886056900024414
99%  : 9.698969841003418

COLD-PAIR
Number of samples : 31839
Minimum y         : 3.0
Maximum y         : 13.0
Mean y            : 6.6395927586707195
Median y    

In [3]:
import os
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
import esm

from torch_geometric.nn import GCNConv, global_mean_pool

DATA_DIR = r"C:\DTI_GNN\data"
CHECKPOINT_DIR = r"C:\DTI_GNN\checkpoints"

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Load target sequences
with open(
    os.path.join(DATA_DIR, "target_sequences.pkl"),
    "rb"
) as f:
    target_sequences = pickle.load(f)

# Load ESM-2 35M
print("Loading ESM-2 35M...")

esm_model, alphabet = esm.pretrained.esm2_t12_35M_UR50D()

batch_converter = alphabet.get_batch_converter()

# Same configuration as training
for p in esm_model.parameters():
    p.requires_grad = False

last_layer_idx = len(esm_model.layers) - 1

for p in esm_model.layers[last_layer_idx].parameters():
    p.requires_grad = True

esm_model = esm_model.to(device)

print("ESM loaded.")

Device: cuda
Loading ESM-2 35M...
ESM loaded.


In [4]:
class DrugEncoder(nn.Module):

    def __init__(
        self,
        in_channels,
        hidden_channels=128,
        out_channels=128
    ):
        super().__init__()

        self.conv1 = GCNConv(
            in_channels,
            hidden_channels
        )

        self.conv2 = GCNConv(
            hidden_channels,
            hidden_channels
        )

        self.conv3 = GCNConv(
            hidden_channels,
            out_channels
        )

    def forward(
        self,
        x,
        edge_index,
        batch
    ):
        x = F.relu(
            self.conv1(x, edge_index)
        )

        x = F.relu(
            self.conv2(x, edge_index)
        )

        x = self.conv3(
            x,
            edge_index
        )

        return global_mean_pool(
            x,
            batch
        )


class TargetEncoder(nn.Module):

    def __init__(
        self,
        esm_model,
        batch_converter,
        target_sequences,
        hidden_dim=128,
        esm_dim=480
    ):
        super().__init__()

        self.esm_model = esm_model
        self.batch_converter = batch_converter
        self.target_sequences = target_sequences

        self.proj = nn.Sequential(
            nn.Linear(esm_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(256, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU()
        )

    def forward(
        self,
        target_idx_batch,
        device
    ):
        unique_idx, inverse = torch.unique(
            target_idx_batch,
            return_inverse=True
        )

        seqs = [
            (
                str(i.item()),
                self.target_sequences[i.item()]
            )
            for i in unique_idx
        ]

        _, _, tokens = self.batch_converter(seqs)

        tokens = tokens.to(device)

        results = self.esm_model(
            tokens,
            repr_layers=[12],
            return_contacts=False
        )

        reps = results["representations"][12]

        pooled = torch.stack([
            reps[
                i,
                1:len(seq) + 1
            ].mean(0)
            for i, (_, seq)
            in enumerate(seqs)
        ])

        projected_unique = self.proj(pooled)

        return projected_unique[inverse]


class DTIModelV3(nn.Module):

    def __init__(
        self,
        drug_in_channels,
        esm_model,
        batch_converter,
        target_sequences,
        hidden_dim=128
    ):
        super().__init__()

        self.drug_encoder = DrugEncoder(
            drug_in_channels,
            hidden_dim,
            hidden_dim
        )

        self.target_encoder = TargetEncoder(
            esm_model,
            batch_converter,
            target_sequences,
            hidden_dim
        )

        self.fusion = nn.Sequential(
            nn.Linear(
                hidden_dim * 2,
                hidden_dim
            ),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(
                hidden_dim,
                64
            ),
            nn.ReLU(),

            nn.Linear(
                64,
                1
            )
        )

    def forward(
        self,
        x,
        edge_index,
        batch,
        target_idx,
        device
    ):
        drug_vec = self.drug_encoder(
            x,
            edge_index,
            batch
        )

        target_vec = self.target_encoder(
            target_idx,
            device
        )

        combined = torch.cat(
            [drug_vec, target_vec],
            dim=1
        )

        return self.fusion(
            combined
        ).squeeze(-1)

In [ ]:
model = DTIModelV3(
    drug_in_channels=18,
    esm_model=esm_model,
    batch_converter=batch_converter,
    target_sequences=target_sequences,
    hidden_dim=128
).to(device)

checkpoint_path = os.path.join(
    CHECKPOINT_DIR,
    "dti_model_v3_stable_epoch15.pt"
)

ckpt = torch.load(
    checkpoint_path,
    map_location=device,
    weights_only=False
)

if ckpt.get("architecture_version") != 2:
    raise ValueError("Checkpoint is from the unstable v3 architecture; retrain with train_v3.py.")

model.load_state_dict(
    ckpt["model_state_dict"]
)

model.eval()

print(
    f"Loaded epoch {ckpt['epoch']}"
)

print(
    f"Train loss: {ckpt['train_loss']:.6f}"
)

In [6]:
EVAL_DIR = r"C:\DTI_GNN\evaluation"

cold_drug = torch.load(
    os.path.join(
        EVAL_DIR,
        "cold_drug_data_v2.pt"
    ),
    weights_only=False
)

print(
    "Cold-drug samples:",
    len(cold_drug)
)

Cold-drug samples: 138556


In [ ]:
from torch_geometric.loader import DataLoader
import numpy as np

def diagnose_predictions(
    model,
    data_list,
    device,
    name,
    n_samples=256
):

    print("\n" + "=" * 80)
    print(f"DIAGNOSTIC: {name}")
    print("=" * 80)

    model.eval()

    sample = data_list[:n_samples]

    loader = DataLoader(
        sample,
        batch_size=len(sample),
        shuffle=False
    )

    batch = next(
        iter(loader)
    ).to(device)

    with torch.no_grad():

        pred = model(
            batch.x,
            batch.edge_index,
            batch.batch,
            batch.target_idx,
            device
        )

    pred = pred.cpu().numpy()

    y = batch.y.cpu().numpy()

    errors = np.abs(
        pred - y
    )

    print("\nTARGET statistics")
    print("-" * 50)

    print("Min     :", y.min())
    print("Max     :", y.max())
    print("Mean    :", y.mean())
    print("Median  :", np.median(y))
    print("Std     :", y.std())

    print("\nPREDICTION statistics")
    print("-" * 50)

    print("Min     :", pred.min())
    print("Max     :", pred.max())
    print("Mean    :", pred.mean())
    print("Median  :", np.median(pred))
    print("Std     :", pred.std())

    print("\nERROR statistics")
    print("-" * 50)

    print("Min     :", errors.min())
    print("Max     :", errors.max())
    print("Mean    :", errors.mean())
    print("Median  :", np.median(errors))

    print("\nFirst 20 predictions vs targets")
    print("-" * 70)

    for i in range(
        min(20, len(pred))
    ):

        print(
            f"{i:3d} | "
            f"Actual = {y[i]:8.4f} | "
            f"Predicted = {pred[i]:12.4f} | "
            f"Error = {errors[i]:12.4f}"
        )

    print("\nLargest 10 prediction errors")
    print("-" * 70)

    worst = np.argsort(
        errors
    )[-10:][::-1]

    for i in worst:

        print(
            f"Index {i:4d} | "
            f"Actual = {y[i]:8.4f} | "
            f"Predicted = {pred[i]:12.4f} | "
            f"Error = {errors[i]:12.4f}"
        )

    return pred, y

In [ ]:
pred_debug, y_debug = diagnose_predictions(
    model,
    cold_drug,
    device,
    "COLD-DRUG",
    n_samples=256
)

In [ ]:
# Load training data
train_data = torch.load(
    r"C:\DTI_GNN\data\train_data_v2.pt",
    weights_only=False
)

print("Training samples:", len(train_data))

# Run exactly the same diagnostic
pred_train_debug, y_train_debug = diagnose_predictions(
    model,
    train_data,
    device,
    "TRAINING DATA",
    n_samples=256
)